# Assignment 2: Milestone I Natural Language Processing
## Task 1. Basic Text Pre-processing
#### Student Name: Nguyen Khac Hieu
#### Student ID: s4099933


Environment: Python 3 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* re
* collections 

## Introduction
This notebook is the implementation of Task 1 in the NLP pipeline process of the cosmetics and beauty products reviews dataset. The aim is to process the original texts to clean the data before using it as an input for the next two tasks (feature representation and classification). The preprocessing methods used include tokenization, removal of stopwords, and filtering by frequencies.

## Importing libraries 

In [1]:
# Importing required libraries for data loading, regex tokenization, and frequency counting
import pandas as pd
import re
from collections import Counter

### 1.1 Examining and loading data
- Examine the data and explain your findings
- Load the data into proper data structures and get it ready for processing.

In [2]:
# I load the full dataset and keep the default integer index
# review_id indexing is not required until Task 2
df = pd.read_csv('cosmetics_beauty_products_reviews.csv')

# Quick structural check — confirms row count, column names, and data looks as expected
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Preview the three columns I will use across all tasks
# review_text is the main preprocessing target for Task 1
print(df[['review_title', 'review_text', 'is_a_buyer']].head(3))

Dataset shape: (61284, 15)
Columns: ['product_id', 'brand_name', 'review_id', 'review_title', 'review_text', 'author', 'review_date', 'review_rating', 'is_a_buyer', 'product_title', 'price', 'avg_product_rating', 'product_rating_count', 'product_tags', 'product_url']
                          review_title  \
0                 Worth buying 50g one   
1           Best cream to start ur day   
2  perfect for summers dry for winters   

                                         review_text  is_a_buyer  
0  Works as it claims. Could see the difference f...        True  
1  It does what it claims . Best thing is it smoo...        True  
2  I have been using this product for months now....        True  


The dataset contains 61,284 reviews across 15 columns. The three columns relevant to this task are review_text (the main text I will preprocess), review_title (used in later tasks for feature augmentation), and is_a_buyer (the classification label for Task 3). Some reviews may have missing text which I handle during tokenization by returning an empty token list.

### 1.2 Pre-processing data
Perform the required text pre-processing steps.

I apply the following preprocessing steps in order: tokenization using the specified regex, lowercasing, removal of short words (length < 2), stopword removal, removal of words with term frequency of 1, and removal of the top 20 most frequent words by document frequency.

In [3]:
# The regex pattern is specified by the assignment and must not be changed
# [a-zA-Z]+ matches one or more letters
# (?:[-'][a-zA-Z]+)? optionally extends the match to include hyphenated words
# like 'well-known' or contractions like "it's" as a single token
# This avoids splitting them into meaningless fragments
TOKENIZE_RE = re.compile(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?")

def tokenize(text):
    # Some reviews are missing (NaN) — return empty list to avoid errors
    # rather than letting the regex crash on a non-string value
    if pd.isna(text):
        return []
    # findall returns all non-overlapping matches as a list of strings
    # lowercase is applied here so all subsequent steps work on normalised text
    return TOKENIZE_RE.findall(text.lower())

df['tokens'] = df['review_text'].apply(tokenize)

# iloc[0] accesses first row by position — df['tokens'][0] would fail
# because the default index label may not start at 0 after any row operations
print(f"Sample tokens from first review: {df['tokens'].iloc[0][:10]}")

Sample tokens from first review: ['works', 'as', 'it', 'claims', 'could', 'see', 'the', 'difference', 'from', 'the']


In [4]:
# Words shorter than 2 characters (single letters like 'a', 'i', 'x')
# carry no semantic meaning and would add noise to the vocabulary
df['tokens'] = df['tokens'].apply(
    lambda toks: [t for t in toks if len(t) >= 2]
)
print(f"Sample after length filter: {df['tokens'].iloc[0][:10]}")

Sample after length filter: ['works', 'as', 'it', 'claims', 'could', 'see', 'the', 'difference', 'from', 'the']


In [5]:
# I load the provided stopword list and remove any token that appears in it
# Stopwords like 'the', 'and', 'is' are too generic to carry meaning
with open('stopwords_en.txt', 'r') as f:
    stopwords = set(f.read().splitlines())

print(f"Total stopwords loaded: {len(stopwords)}")

df['tokens'] = df['tokens'].apply(
    lambda toks: [t for t in toks if t not in stopwords]
)
print(f"Sample after stopword removal: {df['tokens'].iloc[0][:10]}")

Total stopwords loaded: 570
Sample after stopword removal: ['works', 'claims', 'difference', 'day', 'olay', 'cleanser', 'results']


In [6]:
# Term frequency = total number of times a word appears across all reviews combined
# A word appearing only once in ~61,000 reviews is too rare to be a useful feature
# It is more likely a typo, brand-specific jargon, or a one-off word
all_tokens = [tok for toks in df['tokens'] for tok in toks]
term_freq = Counter(all_tokens)

once_words = {word for word, count in term_freq.items() if count == 1}
print(f"Words with term frequency of 1 (to be removed): {len(once_words)}")

df['tokens'] = df['tokens'].apply(
    lambda toks: [t for t in toks if t not in once_words]
)

Words with term frequency of 1 (to be removed): 7734


In [7]:
# Document frequency = number of reviews a word appears in (not total occurrences)
# I use set(toks) per review so a word repeated 5 times in one review
# still only counts as 1 document for that review
# The top 20 most widespread words appear in so many reviews they are
# essentially domain-specific stopwords for cosmetics (e.g. 'skin', 'product')
doc_freq = Counter()
for toks in df['tokens']:
    doc_freq.update(set(toks))

top20 = {word for word, _ in doc_freq.most_common(20)}
print(f"Top 20 by document frequency (removing): {top20}")

df['tokens'] = df['tokens'].apply(
    lambda toks: [t for t in toks if t not in top20]
)

Top 20 by document frequency (removing): {'colour', 'buy', 'nice', 'great', 'hair', 'amazing', 'shade', 'beautiful', 'skin', 'loved', 'nykaa', 'long', 'easy', 'time', 'product', 'color', 'love', 'perfect', 'good', 'smooth'}


## Saving required outputs
Save the requested information as per specification.
- vocab.txt

In [8]:
df['processed_review'] = df['tokens'].apply(lambda toks: ' '.join(toks))

# I save only the columns needed downstream — review_id for Task 2 output 
# formatting, review_title and review_text for Task 3 feature experiments,
# is_a_buyer as the classification label, brand_name/product_title/
# avg_product_rating/price for Task 3 Q2 extra features, and 
# processed_review as the cleaned text output of this task
cols_to_save = ['review_id', 'review_title', 'review_text', 'is_a_buyer',
                'brand_name', 'product_title', 'avg_product_rating', 
                'price', 'processed_review']
df[cols_to_save].to_csv('processed.csv', index=False)
print("processed.csv saved successfully")

processed.csv saved successfully


In [9]:
# Build the final vocabulary from all tokens remaining after every filter step
# sorted() ensures alphabetical order as required by the assignment
# enumerate() provides the integer index starting from 0
# Format is word:index with one entry per line as specified
vocab = sorted(set(tok for toks in df['tokens'] for tok in toks))
print(f"Final vocabulary size: {len(vocab)}")

with open('vocab.txt', 'w') as f:
    for idx, word in enumerate(vocab):
        f.write(f"{word}:{idx}\n")

print("vocab.txt saved successfully")
print("First 5 vocab entries:")
for i, word in enumerate(vocab[:5]):
    print(f"  {word}:{i}")

Final vocabulary size: 8054
vocab.txt saved successfully
First 5 vocab entries:
  aa:0
  aback:1
  abd:2
  abh:3
  ability:4


## Summary
The current task involved building an end-to-end text preprocessing pipeline for the cosmetics reviews dataset. Given the raw text data in the form of reviews, the process began with tokenization based on the given regex, then conversion of the tokens to lowercase letters, followed by filtering based on minimum word length, removal of stop-words, rare words, top 20 common words by their document frequency, resulting in a final vocabulary of 8054 unique words, sorted alphabetically and indexed from 0. The result was stored in the processed.csv and vocab.txt files for use in Task 2.